# Hugging Face Playground

This notebook demonstrates working with Hugging Face models locally and via the API. It covers text generation, audio transcription, and text-to-speech.

## 1) Setup & Install Dependencies

Install required packages for Hugging Face `transformers`, `whisper`, and utilities like `yt-dlp` for downloading videos.

In [ ]:
# Install required libraries (run once)
# Note: This may take a few minutes.
# !pip install -q transformers diffusers accelerate safetensors "huggingface_hub[cli]" youtube-dl yt-dlp torch torchvision torchaudio openai-whisper einops


## 2) Authenticate with Hugging Face (Optional)

If you want to use the Hugging Face API or access private models, set your `HF_TOKEN` as an environment variable.

> **Tip:** You can create/get a token at https://huggingface.co/settings/tokens.

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify token (optional)
print("HF_TOKEN set?", "HF_TOKEN" in os.environ)

HF_TOKEN set? False


## 3) Text Generation (Local)

Use the `transformers` pipeline to generate text from a local model (e.g., `gpt2`).

You can swap in any other compatible model from the Hugging Face Hub.

In [6]:
from transformers import pipeline

# Create a text-generation pipeline (will download model weights the first time)
text_gen = pipeline("text-generation", model="gpt2", device=-1)  # set device=0 for GPU

prompt = "Once upon a time in a world where AI could"
outputs = text_gen(prompt, max_length=120, do_sample=True, temperature=0.8, num_return_sequences=1)

print(outputs[0]["generated_text"])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2161.15it/s]
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=120) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Once upon a time in a world where AI could not communicate with humans, it could not communicate with humans. If you've spent the last half of the 20th century, you know that AI could not communicate with humans. It could not communicate with humans. And if you've lived to 100 years of age, with the next generation of AI, the same thing about human behavior will be the same.

CJ: Do you think it's possible to create a society where machines are the ones saying, "We can't communicate with you while you're on your feet"?

RPS: It will be a lot more difficult to create a society where humans are the ones saying, "Let's try." And the answer is, what we need to do is to do that with machines. That is one of the things I'm pretty excited about.

[Laughs]

CJ: How do you see a society that's very decentralized?

RPS: I think we've seen what kind of ecosystem we've got right through the last 15 years. All of the infrastructure, the internet, the infrastructure, the infrastructure, the infrastr

## 4) Audio Transcription (Whisper)

Download a short YouTube video (via `yt-dlp`) and transcribe it using `openai-whisper`.

In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import yt_dlp
from pathlib import Path

yt_url = "https://youtu.be/5tiLf_bCClU?si=f8IB52ivA1V2rg3q"
out_path = Path("downloaded_video.mp4")

ydl_opts = {
    "format": "bestaudio[ext=m4a]/bestaudio",
    "outtmpl": str(out_path),
    "nocheckcertificate": True,
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([yt_url])

print(f"Downloaded to {out_path}")

import whisper

model = whisper.load_model("small")
result = model.transcribe(str(out_path))
print("Transcription (first 500 chars):")
print(result["text"][:500])

[youtube] Extracting URL: https://youtu.be/5tiLf_bCClU?si=f8IB52ivA1V2rg3q
[youtube] 5tiLf_bCClU: Downloading webpage


[youtube] 5tiLf_bCClU: Downloading android vr player API JSON
[info] 5tiLf_bCClU: Downloading 1 format(s): 140
[download] downloaded_video.mp4 has already been downloaded
[download] 100% of    3.51MiB
Downloaded to downloaded_video.mp4


c:\Users\meirb\OneDrive\Python\project1\.venv-1\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription (first 500 chars):
 Bello! Just another year, couldn't know how you are the best Gonna love you, gonna love you for the rest Make all times always smile on you May happiness always come your way We're here to love you, we're here to celebrate your day We wish the best, oh what the best Can be all new bands in which for you That are your kids known truth Happy birthday, happy birthday to you Happy birthday just for you Happy birthday, happy birthday to you Happy birthday, happy birthday The years may come, can the 


## 5) Text-to-Speech (TTS)

Generate speech audio from text using `facebook/mms-tts-eng`.

Feel free to replace it with another TTS model from the Hub.

In [2]:
from transformers import pipeline
import scipy.io.wavfile

tts = pipeline("text-to-speech", model="facebook/mms-tts-eng", device="cpu")

speech = tts("Hi! I am a text-to-speech model that can convert text into speech.")

output_path = "hf_play_tts.wav"
scipy.io.wavfile.write(output_path, rate=speech["sampling_rate"], data=speech["audio"])
print(f"Saved TTS output to {output_path}")

c:\Users\meirb\OneDrive\Python\project1\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\meirb\OneDrive\Python\project1\.venv-1\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\meirb\.cache\huggingface\hub\models--facebook--mms-tts-eng. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In

Saved TTS output to hf_play_tts.wav
